# NB13

Rebuild revision figure panels from pipeline outputs.

In [ ]:
# Rebuild figure panels

import os, glob, time
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.stats as stats

import matplotlib
matplotlib.rcParams.update({
    "font.family":"Arial","font.size":8,"axes.titlesize":10,"axes.labelsize":8,
    "xtick.labelsize":7,"ytick.labelsize":7,"legend.fontsize":7,"figure.dpi":150,
    "savefig.dpi":1200,"savefig.bbox":"tight","savefig.pad_inches":0.06,
    "axes.linewidth":0.8,"pdf.fonttype":42,"ps.fonttype":42,
})
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
try:
    import seaborn as sns; sns.set_style("ticks"); HAS_SNS=True
except ImportError:
    HAS_SNS=False

BASE = Path(os.environ.get("MES_BASE_DIR", "."))
REV=(BASE/"Manuscript data"/"Tables"/"Revision")
if not REV.exists(): REV=(BASE/"Manuscript Data"/"Tables"/"Revision")
FIGREV=(BASE/"Manuscript data"/"Figures"/"Revision")
if not (BASE/"Manuscript data").exists(): FIGREV=(BASE/"Manuscript Data"/"Figures"/"Revision")
PANELS=FIGREV/"Revision_Panels_PDF"; PANELS.mkdir(parents=True, exist_ok=True)

def find(*names):
    for n in names:
        hits=list(REV.glob(n))
        if hits: return hits[0]
    return None
def rd(p, sheet=0): return pd.read_excel(p, sheet_name=sheet)
def save_panel(fig, name):
    pdf=PANELS/f"{name}.pdf"
    fig.savefig(pdf, bbox_inches="tight")   # vector PDF only
    plt.close(fig)
    print(f"  saved {pdf.name}")
def stars(p):
    if not np.isfinite(p): return ""
    if p<1e-3: return "***"
    if p<1e-2: return "**"
    if p<5e-2: return "*"
    return "ns"

def banner(t): print("\n"+"="*64+f"\n{t}\n"+"="*64)
print(f"Reading data from: {REV}")
print(f"Writing panels to: {PANELS}")

# Color conventions (match existing figures)
C_THY="#C0392B"      # thymus highlight (deep red)
C_CTL="#5B8DB8"      # control tissues (blue)
C_NEG="#4393C3"      # negative coupling blue
C_BAND="#B0B0B0"     # small-effect band grey
C_POS="#D6604D"

# PANEL 1: tissue specificity BOX plot
banner("Figure 2F: tissue specificity (box)")
f=find("Supp_Rev_S1_MultiTissue_Coupling.xlsx")
if f:
    df=rd(f,"coupling_all_tissues")
    df=df.copy(); df["abs_r"]=pd.to_numeric(df["r"],errors="coerce").abs()
    df=df.dropna(subset=["abs_r"])
    # order tissues by mean |r| descending, thymus first if tie
    order=df.groupby("module_source")["abs_r"].mean().sort_values(ascending=False)
    tissues=list(order.index)
    # ensure thymus present; build data per tissue
    data=[df[df["module_source"]==t]["abs_r"].values for t in tissues]
    means=[np.mean(d) for d in data]
    # thymus mean for fold annotation
    thy_mean=df[df["module_source"]=="thymus"]["abs_r"].mean()

    fig,ax=plt.subplots(figsize=(5.6,3.6))
    positions=np.arange(len(tissues))
    bp=ax.boxplot(data, positions=positions, widths=0.6, patch_artist=True,
                  showfliers=False, medianprops=dict(color="black",linewidth=1.0),
                  whiskerprops=dict(color="black",linewidth=0.7),
                  capprops=dict(color="black",linewidth=0.7),
                  boxprops=dict(linewidth=0.7))
    for i,(box,t) in enumerate(zip(bp['boxes'],tissues)):
        box.set_facecolor(C_THY if t=="thymus" else C_CTL)
        box.set_alpha(0.55 if t!="thymus" else 0.85)
    # overlay individual module points (jitter)
    rng=np.random.RandomState(42)
    for i,d in enumerate(data):
        xs=positions[i]+rng.uniform(-0.16,0.16,size=len(d))
        ax.scatter(xs,d,s=8,color="black",alpha=0.45,zorder=3,linewidths=0)
    # fold + significance vs thymus (Mann-Whitney one-sided thymus>control)
    thy=df[df["module_source"]=="thymus"]["abs_r"].values
    ytop=max([np.max(d) for d in data])*1.02
    for i,t in enumerate(tissues):
        if t=="thymus": continue
        ctl=data[i]
        try:
            U,p=stats.mannwhitneyu(thy,ctl,alternative="greater")
            fold=thy_mean/np.mean(ctl) if np.mean(ctl)>0 else np.nan
            ax.text(positions[i], ytop, f"{fold:.1f}x\n{stars(p)}", ha="center",va="bottom",
                    fontsize=6.0, color="black")
        except Exception: pass
    # nice tissue labels
    lab_map={"thymus":"Thymus","blood":"Blood","spleen":"Spleen","liver":"Liver",
             "lung":"Lung","bone_marrow":"Bone marrow","lymph_node":"Lymph node",
             "ALL_nonthymus_pooled":"Pooled"}
    ax.set_xticks(positions)
    ax.set_xticklabels([lab_map.get(t,t) for t in tissues], rotation=30, ha="right")
    ax.set_ylabel("|MES-tolerance coupling| (Spearman |r|)")
    ax.set_ylim(0, ytop*1.18)
    ax.set_title("Thymus-derived modules couple more strongly than other tissues")
    for s in ["top","right"]: ax.spines[s].set_visible(False)
    leg=[Patch(facecolor=C_THY,alpha=0.85,label="Thymus (source)"),
         Patch(facecolor=C_CTL,alpha=0.55,label="Control tissue")]
    ax.legend(handles=leg, loc="upper right", frameon=False, fontsize=6.5)
    save_panel(fig,"Figure_2F_tissue_specificity")
else:
    print("  MISSING Supp_Rev_S1_MultiTissue_Coupling.xlsx")

# PANEL 2: GR interaction FOREST (all 8 x 3 = 24 rows)
banner("Figure 3D: GR interaction forest (8x3, replaces old 3D)")
parts=[]
fN=find("Supp_Rev_N2_Canonical_GR_Interaction.xlsx")
if fN:
    d=rd(fN,0).copy(); d["cohort"]=d["dataset"].astype(str)
    parts.append(d)
fP=find("Supp_Rev_P2_External_GR_Interaction.xlsx")
if fP:
    d=rd(fP,0).copy(); d["cohort"]=d["dataset"].astype(str)
    parts.append(d)
if parts:
    g=pd.concat(parts, ignore_index=True, sort=False)
    # need beta_int, ci_lo, ci_hi, MES, cohort, q_BH, above_small_effect
    for c in ["beta_int","ci_lo","ci_hi","q_BH"]:
        if c in g.columns: g[c]=pd.to_numeric(g[c],errors="coerce")
    g=g.dropna(subset=["beta_int","ci_lo","ci_hi"])
    # order: cohort groups (Olah, SEA-AD, GSE174367), MES01..08 within
    cohort_order=[c for c in ["Olah","SEA-AD","GSE174367"] if c in g["cohort"].unique()]
    cohort_order+=[c for c in g["cohort"].unique() if c not in cohort_order]
    rows=[]
    for ch in cohort_order:
        sub=g[g["cohort"]==ch].copy()
        sub=sub.sort_values("MES")
        for _,r in sub.iterrows(): rows.append(r)
    gg=pd.DataFrame(rows).reset_index(drop=True)
    n=len(gg)
    SMALL=0.10
    fig,ax=plt.subplots(figsize=(5.8, max(4.0, 0.26*n)))
    yy=np.arange(n)[::-1]  # top-to-bottom
    # subtle shaded band marking the |beta| < 0.10 negligible-effect bound,
    # with boundary lines annotated directly so no legend entry is needed.
    ax.axvspan(-SMALL, SMALL, color="#F2F5F9", alpha=1.0, zorder=0)
    ax.axvline(-SMALL, color="#A8B4C0", lw=0.9, ls=(0,(4,2)), zorder=1)
    ax.axvline( SMALL, color="#A8B4C0", lw=0.9, ls=(0,(4,2)), zorder=1)
    ax.axvline(0, color="#333333", lw=1.0, ls="-", zorder=1)
    # points + CI
    for i,(_,r) in enumerate(gg.iterrows()):
        y=yy[i]; b=r["beta_int"]; lo=r["ci_lo"]; hi=r["ci_hi"]
        big=bool(r.get("above_small_effect",False))
        col = "#C0392B" if big else "#2C5F8A"   # navy = within bound; red only if exceeds
        ax.plot([lo,hi],[y,y], color="#6B7B8C", lw=1.0, zorder=2,
                solid_capstyle="round")
        ax.plot(b,y,"o",ms=4.8, color=col, mec="white", mew=0.6, zorder=3)
    # labels: MES per row, cohort as group separators
    ylabels=[f"{r['MES']}" for _,r in gg.iterrows()]
    ax.set_yticks(yy); ax.set_yticklabels(ylabels, fontsize=6.0)
    # cohort group brackets on the left
    # compute spans
    start=0
    trans=ax.get_yaxis_transform()
    for ch in cohort_order:
        idx=[i for i,(_,r) in enumerate(gg.iterrows()) if r["cohort"]==ch]
        if not idx: continue
        ytop=yy[idx[0]]+0.5; ybot=yy[idx[-1]]-0.5
        ax.text(-0.18, (ytop+ybot)/2, ch, transform=trans, ha="right", va="center",
                fontsize=7.5, fontweight="bold", rotation=90)
    # x-limits: at least +/-0.12 so the +/-0.10 band reads as a bounded zone,
    # widened to data if CIs extend further
    xmax=max(0.13, float(np.nanmax(np.abs(np.r_[gg["ci_lo"].values, gg["ci_hi"].values])))*1.10)
    ax.set_xlim(-xmax, xmax)
    # annotate the negligible-effect bound directly above the band edges
    ytop_ann=len(gg)-0.2
    ax.annotate('|β| < 0.10 (negligible-effect zone)', xy=(0, ytop_ann),
                ha="center", va="bottom", fontsize=6.0, color="#5A6B7B",
                annotation_clip=False)
    ax.set_xlabel("Standardized MES x GR interaction coefficient (95% CI)")
    ax.set_title("No module shows non-trivial GR moderation (all |β|<0.10)")
    for s in ["top","right","left"]: ax.spines[s].set_visible(False)
    ax.tick_params(axis="y", length=0)
    # legend describes ONLY the dots (each is a point you can match). The band's
    # meaning is annotated on the axis above, so it needs no legend entry.
    any_exceed=bool(gg.get("above_small_effect", pd.Series([False]*len(gg))).any())
    leg=[plt.Line2D([0],[0], marker="o", linestyle="none", mfc="#2C5F8A", mec="white", mew=0.5,
                    ms=6, label="MES x GR interaction estimate (95% CI)")]
    if any_exceed:
        leg.append(plt.Line2D([0],[0], marker="o", linestyle="none", mfc="#C0392B", mec="white",
                              mew=0.5, ms=6, label="estimate exceeding |β| = 0.10"))
    ax.legend(handles=leg, loc="lower right", frameon=True, framealpha=0.9,
              edgecolor="#CCCCCC", fontsize=6.2)
    save_panel(fig,"Figure_3D_GR_interaction_forest")
else:
    print("  MISSING N2/P2 GR interaction files")

# PANEL 3: purity dose-response
banner("Figure 2E: purity dose-response")
f=find("Supp_Rev_A10_Microglia_Purity.xlsx")
if f:
    strat=rd(f,"MES_vs_Tol_by_Purity")
    # mean coupling across modules, per cohort, per purity quartile
    qcol="purity_quartile"
    strat["q_order"]=strat[qcol].astype(str).str.extract(r"(\d)").astype(float)
    g=strat.groupby(["dataset","q_order"])["r"].mean().reset_index()
    fig,ax=plt.subplots(figsize=(5.0,3.4))
    cohorts=sorted(g["dataset"].unique())
    cmap=plt.cm.viridis(np.linspace(0,0.85,len(cohorts)))
    for c,col in zip(cohorts,cmap):
        sub=g[g["dataset"]==c].sort_values("q_order")
        ax.plot(sub["q_order"], sub["r"], "-o", ms=4, lw=1.2, color=col, label=str(c))
    ax.axhline(0, color="grey", lw=0.5, ls=":")
    ax.set_xticks([1,2,3,4]); ax.set_xticklabels(["Q1\n(low)","Q2","Q3","Q4\n(high)"])
    ax.set_xlabel("Microglial purity quartile")
    ax.set_ylabel("Mean MES-tolerance coupling (adjusted ρ)")
    ax.set_title("Coupling strengthens with microglial purity")
    for s in ["top","right"]: ax.spines[s].set_visible(False)
    ax.legend(loc="upper right", frameon=False, fontsize=6.0, ncol=1)
    save_panel(fig,"Figure_2E_purity_dose_response")
else:
    print("  MISSING Supp_Rev_A10_Microglia_Purity.xlsx")

banner("DONE")
print(f"Panels in: {PANELS}")
print("Each panel is a standalone vector PDF. Assemble into composite figures as you like.")
